# Data Science – Pertemuan 7

## Identitas Mahasiswa

**Nama:** Amin Siddik Rangkuti  
**NIM:** 220401010124  
**Kelas:** IF405  
**Program Studi:** PJJ Informatika  
**Mata Kuliah:** Data Science  

---

## Topik Praktikum
Pengantar Machine Learning: Klasifikasi, Evaluasi Model, dan Prediksi Data Baru

## Tujuan Praktikum
Pada praktikum ini dilakukan penerapan Machine Learning sederhana menggunakan dataset Iris. Praktikum mencakup proses eksplorasi data, preprocessing, pembagian data, pelatihan model, evaluasi performa model, visualisasi hasil, dan prediksi data baru.

## Library yang Digunakan
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn


<a href="https://colab.research.google.com/github/aminsiddik2810/data-science-2026/blob/main/Pertemuan_7_AminSiddikRangkuti_220401010124.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Library

Pada tahap ini dilakukan import library Python yang diperlukan untuk pengolahan data, visualisasi, preprocessing, pembuatan model Machine Learning, dan evaluasi model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# 2. Memuat Dataset Iris

Dataset Iris digunakan sebagai contoh dataset klasifikasi. Dataset ini memiliki empat fitur numerik, yaitu panjang dan lebar sepal serta panjang dan lebar petal. Target klasifikasinya adalah jenis spesies bunga Iris.

In [ ]:
iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target
df["species"] = df["target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

df.head()

# 3. Informasi Awal Dataset

Tahap ini digunakan untuk mengetahui ukuran dataset, tipe data, nama kolom, serta beberapa data awal.

In [ ]:
print("Ukuran dataset:", df.shape)

print("\nNama kolom:")
print(df.columns.tolist())

print("\nInformasi dataset:")
df.info()

print("\nData awal:")
display(df.head())

# 4. Statistik Deskriptif

Statistik deskriptif digunakan untuk melihat gambaran umum data numerik seperti mean, standar deviasi, nilai minimum, maksimum, dan kuartil.

In [ ]:
df.describe().round(3)

# 5. Mengecek Missing Value dan Duplikasi

Sebelum membuat model, dataset perlu diperiksa apakah terdapat nilai kosong atau data duplikat yang dapat memengaruhi hasil analisis.

In [ ]:
print("Missing value:")
print(df.isnull().sum())

print("\nJumlah data duplikat:")
print(df.duplicated().sum())

# 6. Distribusi Target

Distribusi target digunakan untuk melihat apakah jumlah data pada setiap kelas spesies seimbang atau tidak.

In [ ]:
target_count = df["species"].value_counts()

display(target_count)

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="species")
plt.title("Distribusi Kelas Species Iris")
plt.xlabel("Species")
plt.ylabel("Jumlah Data")
plt.show()

# 7. Visualisasi Distribusi Fitur

Histogram digunakan untuk memahami sebaran nilai pada setiap fitur numerik.

In [ ]:
feature_cols = iris.feature_names

for col in feature_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, hue="species", kde=True, bins=20)
    plt.title(f"Distribusi {col}")
    plt.xlabel(col)
    plt.ylabel("Frekuensi")
    plt.show()

# 8. Visualisasi Hubungan Antar Fitur

Pairplot digunakan untuk melihat hubungan antar fitur sekaligus melihat pola pemisahan setiap spesies.

In [ ]:
sns.pairplot(df, hue="species", vars=feature_cols)
plt.show()

# 9. Korelasi Antar Fitur

Heatmap korelasi digunakan untuk melihat hubungan antar variabel numerik. Nilai korelasi mendekati 1 menunjukkan hubungan positif yang kuat.

In [ ]:
corr = df[feature_cols].corr()

plt.figure(figsize=(8, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1
)
plt.title("Heatmap Korelasi Fitur Iris")
plt.show()

display(corr.round(3))

# 10. Memisahkan Fitur dan Target

Fitur digunakan sebagai input model, sedangkan target adalah label yang akan diprediksi oleh model.

In [ ]:
X = df[feature_cols]
y = df["target"]

print("Contoh fitur:")
display(X.head())

print("\nContoh target:")
display(y.head())

# 11. Membagi Data Latih dan Data Uji

Dataset dibagi menjadi data latih dan data uji. Data latih digunakan untuk melatih model, sedangkan data uji digunakan untuk mengukur performa model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data latih:", X_train.shape[0])
print("Jumlah data uji:", X_test.shape[0])

# 12. Standarisasi Data

Standarisasi dilakukan agar setiap fitur memiliki skala yang seragam. Hal ini penting terutama untuk algoritma berbasis jarak seperti K-Nearest Neighbors.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_train_scaled_df.head()

# 13. Model 1: K-Nearest Neighbors (KNN)

KNN adalah algoritma klasifikasi yang memprediksi kelas data baru berdasarkan kedekatannya dengan data lain.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)

accuracy_knn = accuracy_score(y_test, y_pred_knn)

print("Akurasi KNN:", round(accuracy_knn, 4))
print("\nClassification Report KNN:")
print(classification_report(y_test, y_pred_knn, target_names=iris.target_names))

# 14. Model 2: Decision Tree

Decision Tree adalah algoritma klasifikasi yang membuat aturan keputusan dalam bentuk struktur pohon.

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("Akurasi Decision Tree:", round(accuracy_dt, 4))
print("\nClassification Report Decision Tree:")
print(classification_report(y_test, y_pred_dt, target_names=iris.target_names))

# 15. Model 3: Random Forest

Random Forest adalah algoritma ensemble yang membangun banyak pohon keputusan dan menggabungkan hasil prediksinya untuk meningkatkan performa model.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Akurasi Random Forest:", round(accuracy_rf, 4))
print("\nClassification Report Random Forest:")
print(classification_report(y_test, y_pred_rf, target_names=iris.target_names))

# 16. Confusion Matrix Setiap Model

Confusion matrix digunakan untuk melihat jumlah prediksi benar dan salah pada setiap kelas.

In [ ]:
models_result = {
    "KNN": y_pred_knn,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf
}

for model_name, y_pred in models_result.items():
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=iris.target_names,
        yticklabels=iris.target_names
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

# 17. Perbandingan Akurasi Model

Pada bagian ini dilakukan perbandingan akurasi dari tiga model Machine Learning yang digunakan.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["KNN", "Decision Tree", "Random Forest"],
    "Accuracy": [accuracy_knn, accuracy_dt, accuracy_rf]
})

display(comparison)

plt.figure(figsize=(8, 4))
sns.barplot(data=comparison, x="Model", y="Accuracy")
plt.title("Perbandingan Akurasi Model")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1.1)
plt.show()

# 18. Visualisasi Decision Tree

Visualisasi pohon keputusan digunakan untuk melihat aturan yang dibuat oleh model Decision Tree.

In [ ]:
plt.figure(figsize=(15, 8))
plot_tree(
    dt_model,
    feature_names=feature_cols,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)
plt.title("Visualisasi Decision Tree")
plt.show()

# 19. Feature Importance Random Forest

Feature importance digunakan untuk mengetahui fitur mana yang paling berpengaruh dalam proses klasifikasi.

In [ ]:
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

display(importance)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# 20. Prediksi Data Baru

Pada tahap ini model digunakan untuk memprediksi spesies Iris berdasarkan data baru yang dimasukkan.

In [ ]:
new_data = pd.DataFrame({
    "sepal length (cm)": [5.1, 6.4, 7.2],
    "sepal width (cm)": [3.5, 3.2, 3.0],
    "petal length (cm)": [1.4, 4.5, 5.8],
    "petal width (cm)": [0.2, 1.5, 2.1]
})

new_data_scaled = scaler.transform(new_data)

pred_knn = knn_model.predict(new_data_scaled)
pred_dt = dt_model.predict(new_data)
pred_rf = rf_model.predict(new_data)

result_prediction = new_data.copy()
result_prediction["Prediksi KNN"] = [iris.target_names[i] for i in pred_knn]
result_prediction["Prediksi Decision Tree"] = [iris.target_names[i] for i in pred_dt]
result_prediction["Prediksi Random Forest"] = [iris.target_names[i] for i in pred_rf]

result_prediction

# 21. Interpretasi Hasil

Berdasarkan hasil evaluasi, model KNN, Decision Tree, dan Random Forest dapat digunakan untuk melakukan klasifikasi spesies bunga Iris. Dataset Iris memiliki pola pemisahan kelas yang cukup jelas, terutama pada fitur `petal length` dan `petal width`.

KNN bekerja berdasarkan kedekatan jarak antar data sehingga membutuhkan standarisasi. Decision Tree menghasilkan aturan yang mudah dipahami dalam bentuk pohon keputusan. Random Forest menggunakan banyak pohon keputusan sehingga dapat memberikan hasil yang lebih stabil.

Feature importance pada Random Forest menunjukkan fitur yang paling berpengaruh terhadap hasil prediksi. Pada dataset Iris, fitur petal umumnya memiliki pengaruh yang kuat dalam membedakan spesies.


# Kesimpulan

Pada praktikum Pertemuan 7, dilakukan penerapan Machine Learning untuk klasifikasi dataset Iris menggunakan tiga algoritma, yaitu K-Nearest Neighbors, Decision Tree, dan Random Forest.

## Apa yang Dipelajari
Pada pertemuan ini saya mempelajari alur dasar pembuatan model Machine Learning, mulai dari memahami dataset, melakukan eksplorasi data, memisahkan fitur dan target, membagi data menjadi data latih dan data uji, melakukan standarisasi, melatih model, mengevaluasi model, membandingkan akurasi, dan melakukan prediksi data baru.

## Temuan Utama
- Dataset Iris memiliki 150 data dan 3 kelas spesies, yaitu setosa, versicolor, dan virginica.
- Fitur petal length dan petal width memiliki hubungan yang kuat dan berperan penting dalam membedakan spesies.
- KNN membutuhkan data yang sudah distandarisasi karena menggunakan konsep jarak.
- Decision Tree mudah dipahami karena menghasilkan aturan dalam bentuk pohon keputusan.
- Random Forest dapat digunakan untuk mengetahui fitur yang paling berpengaruh melalui feature importance.
- Evaluasi menggunakan accuracy, classification report, dan confusion matrix membantu memahami performa model secara lebih lengkap.

## Keterbatasan
Dataset Iris merupakan dataset yang sederhana dan bersih, sehingga hasil model cenderung baik. Pada data nyata, kemungkinan terdapat missing value, outlier, data tidak seimbang, dan fitur yang lebih kompleks sehingga diperlukan preprocessing lebih lanjut.

## Penutup
Praktikum ini memberikan pemahaman penting mengenai konsep dasar Machine Learning dalam Data Science. Proses eksplorasi data, pelatihan model, evaluasi, dan interpretasi hasil merupakan tahapan utama sebelum model dapat digunakan untuk pengambilan keputusan.
